# Promote a model to production

Training a model (notebooks 03-05) only produces a **candidate** in `/app/models/<name>.pkl` plus its `<name>.metrics.json`.
It does **not** change what the API serves.

This notebook is the explicit gate: compare the candidate's metrics against whatever is currently in
`/app/models/production/manifest.json`, decide, and only then promote. The API always serves whatever
is recorded in the manifest, so promoting is the only way to change it.

In [1]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

MODEL_DIR = Path('/app/models')
PRODUCTION_DIR = MODEL_DIR / 'production'
MANIFEST_PATH = PRODUCTION_DIR / 'manifest.json'

PROMOTED_BY = 'juliana'  # change to your own name before promoting

In [2]:
def load_manifest() -> dict:
    if not MANIFEST_PATH.exists():
        return {}
    return json.loads(MANIFEST_PATH.read_text())


def candidate_metrics(model_name: str) -> dict | None:
    path = MODEL_DIR / f'{model_name}.metrics.json'
    if not path.exists():
        return None
    return json.loads(path.read_text())


def compare(model_name: str) -> None:
    manifest = load_manifest()
    production = manifest.get(model_name)
    candidate = candidate_metrics(model_name)

    print(f'--- {model_name} ---')
    if production:
        print('In production :', production['version'], production['metrics'])
    else:
        print('In production : (none yet)')

    if candidate:
        print('Candidate     :', candidate)
    else:
        print('Candidate     : (no candidate trained — run the training notebook first)')


def promote(model_name: str, promoted_by: str = PROMOTED_BY) -> None:
    """Copy the current candidate .pkl into production/ and record it in manifest.json.

    Call this only after reviewing `compare(model_name)` and deciding the candidate
    is the one that should be served — it does not have to be the most recent training run.
    """
    candidate_path = MODEL_DIR / f'{model_name}.pkl'
    metrics = candidate_metrics(model_name)
    if not candidate_path.exists() or metrics is None:
        raise FileNotFoundError(f'No trained candidate for "{model_name}". Run its training notebook first.')

    manifest = load_manifest()
    existing_versions = [
        int(v.removeprefix('v'))
        for v in (PRODUCTION_DIR / model_name).glob('v*')
    ] if (PRODUCTION_DIR / model_name).exists() else []
    version = f'v{max(existing_versions, default=0) + 1}'

    version_dir = PRODUCTION_DIR / model_name / version
    version_dir.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(candidate_path, version_dir / 'model.pkl')

    manifest[model_name] = {
        'version': version,
        'metrics': metrics,
        'promoted_at': datetime.now(timezone.utc).isoformat(),
        'promoted_by': promoted_by,
    }
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
    print(f'Promoted {model_name} {version} to production.')

## 1. Review

Compare the freshly trained candidate against what production is currently serving, for each model type.

In [3]:
for name in ['logistic_regression', 'decision_tree', 'random_forest']:
    compare(name)
    print()

--- logistic_regression ---
In production : v1 {'accuracy': 0.9666666666666667, 'precision': 0.9696969696969696, 'recall': 0.9666666666666667, 'f1': 0.9665831244778613, 'trained_at': None}
Candidate     : {'accuracy': 0.9666666666666667, 'precision': 0.9696969696969696, 'recall': 0.9666666666666667, 'f1': 0.9665831244778613, 'trained_at': None}

--- decision_tree ---
In production : v1 {'accuracy': 0.9333333333333333, 'precision': 0.9333333333333333, 'recall': 0.9333333333333333, 'f1': 0.9333333333333333, 'trained_at': None}
Candidate     : {'accuracy': 0.9333333333333333, 'precision': 0.9333333333333333, 'recall': 0.9333333333333333, 'f1': 0.9333333333333333, 'trained_at': None}

--- random_forest ---
In production : v1 {'accuracy': 0.9, 'precision': 0.9023569023569024, 'recall': 0.9, 'f1': 0.8997493734335839, 'trained_at': None}
Candidate     : {'accuracy': 0.9, 'precision': 0.9023569023569024, 'recall': 0.9, 'f1': 0.8997493734335839, 'trained_at': None}



## 2. Decide

Only run `promote(...)` for a model whose candidate you actually want serving traffic.
Skipping a model here means production keeps whatever version it already has — a newer
but worse candidate never takes over on its own.

In [4]:
# Example: promote random_forest's current candidate after confirming it's better in the comparison above.
# promote('random_forest')

## 3. Verify

Confirm the manifest now reflects what you intended — this is exactly what `GET /models` returns.

In [5]:
print(json.dumps(load_manifest(), indent=2))

{
  "logistic_regression": {
    "version": "v1",
    "metrics": {
      "accuracy": 0.9666666666666667,
      "precision": 0.9696969696969696,
      "recall": 0.9666666666666667,
      "f1": 0.9665831244778613,
      "trained_at": null
    },
    "promoted_at": "2026-09-06T19:00:42.600321+00:00",
    "promoted_by": "bootstrap-migration"
  },
  "decision_tree": {
    "version": "v1",
    "metrics": {
      "accuracy": 0.9333333333333333,
      "precision": 0.9333333333333333,
      "recall": 0.9333333333333333,
      "f1": 0.9333333333333333,
      "trained_at": null
    },
    "promoted_at": "2026-09-06T19:00:42.646908+00:00",
    "promoted_by": "bootstrap-migration"
  },
  "random_forest": {
    "version": "v1",
    "metrics": {
      "accuracy": 0.9,
      "precision": 0.9023569023569024,
      "recall": 0.9,
      "f1": 0.8997493734335839,
      "trained_at": null
    },
    "promoted_at": "2026-09-06T19:00:42.690142+00:00",
    "promoted_by": "bootstrap-migration"
  }
}
